# USED SMARTPHONES PRICES PREDICTION USING PYTHON

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load the dataset

In [ ]:
from pathlib import Path

# Use the CSV next to this notebook. In an interactive notebook without __file__,
# the CSV is read from the current working directory.
DATA_PATH = Path(__file__).with_name("used_device_data.csv") if "__file__" in globals() else Path("used_device_data.csv")
used_phones = pd.read_csv(DATA_PATH)
used_phones.head()

## 3. Review the dataset

In [ ]:
used_phones.shape

In [ ]:
used_phones.info()

## 4. Check missing values

In [ ]:
used_phones.isnull().sum()

Missing numeric and categorical values are handled in the preprocessing pipelines below. This keeps training and prediction preprocessing consistent.

## 5. Exploratory data analysis

### Distribution of normalized used prices

In [ ]:
sns.histplot(data=used_phones, x="normalized_used_price", kde=True)
plt.title("Distribution of normalized used price")
plt.xlabel("Normalized used price")
plt.show()

### Color-coded correlations between numeric features


In [ ]:
numeric_feature_correlations = used_phones.select_dtypes(include="number").corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    numeric_feature_correlations,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"},
)
plt.title("Correlation matrix of numeric features")
plt.tight_layout()
plt.show()

### Relationship between normalized new and used prices

In [ ]:
used_phones[["normalized_used_price", "normalized_new_price"]].corr()

## 6. Price conversion helpers

The source prices are treated as natural-log values in euros. The helper cells below reverse that scale for display and prepare euro inputs for the model. The dataset does not publish a separate conversion formula, so this assumption should be stated when presenting the results.

### Convert normalized model output back to euros

In [ ]:
def normalized_to_real_price(normalized_price):
    """Undo the natural-log transform and return an estimated price in euros."""
    value = float(normalized_price)
    if not np.isfinite(value) or value > 709:
        raise ValueError("Normalized price is outside the supported numeric range.")
    return float(np.exp(value))

### Convert a euro price to the model input scale

In [ ]:
def real_price_to_normalized(real_price_eur):
    """Apply the natural-log transform used by this dataset to a euro price."""
    value = float(real_price_eur)
    if not np.isfinite(value) or value <= 0:
        raise ValueError("Price must be a finite positive amount in euros.")
    return float(np.log(value))

### Make a prediction and convert its output

In [ ]:
def predict_used_price(fitted_model, phone_features):
    """Return the predicted normalized value and its estimated euro price."""
    predicted_normalized = float(fitted_model.predict(phone_features)[0])
    return predicted_normalized, normalized_to_real_price(predicted_normalized)

### Evaluate predictions in euros

In [ ]:
def evaluate_model_real_prices(fitted_model, X_holdout, y_holdout):
    """Evaluate holdout predictions after converting prices back to euros."""
    if len(X_holdout) == 0 or len(y_holdout) == 0:
        raise ValueError("The holdout set is empty.")
    predicted_normalized = np.asarray(fitted_model.predict(X_holdout), dtype=float)
    actual_normalized = np.asarray(y_holdout, dtype=float)
    if not np.isfinite(predicted_normalized).all() or not np.isfinite(actual_normalized).all():
        raise ValueError("Holdout predictions or actual values contain non-finite values.")
    actual_eur = np.asarray([normalized_to_real_price(v) for v in actual_normalized])
    predicted_eur = np.asarray([normalized_to_real_price(v) for v in predicted_normalized])
    metrics = {
        "mae_eur": float(mean_absolute_error(actual_eur, predicted_eur)),
        "rmse_eur": float(np.sqrt(mean_squared_error(actual_eur, predicted_eur))),
        "r2_eur": float(r2_score(actual_eur, predicted_eur)),
    }
    results = X_holdout.copy()
    results["actual_price_eur"] = actual_eur
    results["predicted_price_eur"] = predicted_eur
    results["absolute_error_eur"] = np.abs(actual_eur - predicted_eur)
    return metrics, results

## 7. Prepare features and target

### Select the prediction target and input features

In [ ]:
target = "normalized_used_price"
X = used_phones.drop(columns=[target]).copy()
y = used_phones[target].copy()

### Identify categorical and numeric input columns

In [ ]:
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category", "bool"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

## 8. Build the preprocessing and model pipeline

### Prepare numeric inputs

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

### Prepare categorical inputs

In [ ]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

### Combine preprocessing steps

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols),
    ],
    sparse_threshold=0,
)

## 9. Split the data and train the model

### Create training and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Fit preprocessing on the training rows


In [ ]:
preprocessor.fit(X_train)

### Configure expected price directions
The data shows used prices generally fall as days used increases and rise with battery capacity. The estimator learns the size and shape of these effects from the data, while these two domain rules prevent counterintuitive reversals for otherwise identical phones. No prices or model coefficients are manually assigned.


In [ ]:
model_feature_names = list(preprocessor.get_feature_names_out())
monotonic_constraints = np.zeros(len(model_feature_names), dtype=int)
monotonic_constraints[model_feature_names.index("numeric__days_used")] = -1
monotonic_constraints[model_feature_names.index("numeric__battery")] = 1

### Define the gradient boosting model


In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", HistGradientBoostingRegressor(
        max_iter=100,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0,
        monotonic_cst=monotonic_constraints,
        random_state=42,
    )),
])

### Fit the model

In [ ]:
model.fit(X_train, y_train)

## 10. Evaluate the model

### Generate predictions for the test set

In [ ]:
y_pred = model.predict(X_test)

### Calculate error metrics on the normalized scale

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE (normalized scale):", round(mae, 4))
print("RMSE (normalized scale):", round(rmse, 4))
print("R² score:", round(r2, 4))

### Plot actual versus predicted normalized prices

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual normalized used price")
plt.ylabel("Predicted normalized used price")
plt.title("Actual vs. predicted normalized used price")
plt.tight_layout()
plt.show()

### Calculate error metrics in estimated euros

In [ ]:
euro_metrics, euro_test_results = evaluate_model_real_prices(model, X_test, y_test)
print("MAE (EUR):", round(euro_metrics["mae_eur"], 2))
print("RMSE (EUR):", round(euro_metrics["rmse_eur"], 2))
print("R² score (EUR):", round(euro_metrics["r2_eur"], 4))

## 11. Predict for a new phone

Enter the phone details using the same columns and category spellings as the dataset. Change the example values to match the phone. `normalized_new_price` must be on the model scale; use `real_price_to_normalized()` first if you have the new phone price in euros.

### Enter phone specifications

In [ ]:
new_phone = pd.DataFrame([{
    "device_brand": "Samsung",
    "os": "Android",
    "screen_size": 15.0,
    "4g": "yes",
    "5g": "yes",
    "rear_camera_mp": 48,
    "front_camera_mp": 16,
    "internal_memory": 128,
    "ram": 8,
    "battery": 4500,
    "weight": 190,
    "release_year": 2020,
    "days_used": 565,
    "normalized_new_price": 5.5,
}])

### Align the input columns with the training data

In [ ]:
new_phone = new_phone.reindex(columns=X.columns)
new_phone

### Predict and display the estimated euro price

In [ ]:
predicted_normalized_used_price, predicted_real_price_eur = predict_used_price(model, new_phone)
print("Predicted normalized used price:", round(predicted_normalized_used_price, 4))
print("Estimated used-device price (EUR):", round(predicted_real_price_eur, 2))